# TP3 : Text Mining

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier   

In [ ]:
from sklearn.feature_selection import chi2
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import f_classif

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('E-commerce_Dataset.csv', header=None, names=['Category', 'Description'])
data = data.dropna()
print("Valeurs nulles :\n", data.isnull().sum())
print("Taille du dataset :", data.shape)

Valeurs nulles :
 Category       0
Description    0
dtype: int64
Taille du dataset : (50424, 2)


In [4]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(data['Description'])
print("Shape of CountVectorizer:", X_train_counts.shape)

Shape of CountVectorizer: (50424, 78878)


In [5]:
from sklearn.feature_extraction.text import TfidfTransformer
tf_transformer = TfidfTransformer(use_idf=True).fit(X_train_counts)
X_tfidf = tf_transformer.transform(X_train_counts)
print("Shape of TF-IDF:", X_tfidf.shape)

Shape of TF-IDF: (50424, 78878)


In [6]:
le = LabelEncoder()
y = le.fit_transform(data['Category'])

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (40339, 78878)
X_test shape: (10085, 78878)


In [7]:

selection_methods = {
    'Chi-Square': chi2,
    'Mutual Information / Info Gain': mutual_info_classif,
    'Fisher Score': f_classif
}

classifiers = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(kernel='linear', random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss')
}

results = []
k_features = 1000 

In [8]:
for sel_name, sel_func in selection_methods.items():
    print(f"\n--- Application de la sélection de variables : {sel_name} ---")
    selector = SelectKBest(score_func=sel_func, k=k_features)
    
    X_train_sel = selector.fit_transform(X_train, y_train)
    X_test_sel = selector.transform(X_test)
    
    for clf_name, clf in classifiers.items():
        print(f"Entraînement de {clf_name}...")
        clf.fit(X_train_sel, y_train)
        y_pred = clf.predict(X_test_sel)
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        results.append({
            'Méthode de Sélection': sel_name,
            'Algorithme': clf_name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-score': f1
        })

print("\nProcessus terminé !")


--- Application de la sélection de variables : Chi-Square ---
Entraînement de Decision Tree...
Entraînement de KNN...
Entraînement de SVM...
Entraînement de XGBoost...

--- Application de la sélection de variables : Mutual Information / Info Gain ---
Entraînement de Decision Tree...
Entraînement de KNN...
Entraînement de SVM...
Entraînement de XGBoost...

--- Application de la sélection de variables : Fisher Score ---
Entraînement de Decision Tree...
Entraînement de KNN...
Entraînement de SVM...
Entraînement de XGBoost...

Processus terminé !


In [9]:

df_results = pd.DataFrame(results)
df_results.sort_values(by='F1-score', ascending=False, inplace=True)
df_results.reset_index(drop=True, inplace=True)
display(df_results)

,Méthode de Sélection,Algorithme,Accuracy,Precision,Recall,F1-score
0,Chi-Square,XGBoost,0.950322,0.950439,0.950322,0.950296
1,Mutual Information / Info Gain,XGBoost,0.950223,0.950379,0.950223,0.950190
2,Fisher Score,XGBoost,0.948835,0.948924,0.948835,0.948810
3,Chi-Square,Decision Tree,0.948042,0.948027,0.948042,0.948030
4,Fisher Score,Decision Tree,0.944274,0.944317,0.944274,0.944288
5,Mutual Information / Info Gain,Decision Tree,0.943480,0.943517,0.943480,0.943492
6,Chi-Square,SVM,0.934160,0.934987,0.934160,0.934033
7,Fisher Score,SVM,0.932375,0.933194,0.932375,0.932275
8,Mutual Information / Info Gain,SVM,0.929301,0.929854,0.929301,0.929132
9,Chi-Square,KNN,0.907586,0.910554,0.907586,0.907904
